1.注意力机制中单向和双向，以及交叉的区别是什么
2.什么是多头单向注意力机制

In [19]:
import pandas as pd
import json
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from datasets import Dataset
import gzip
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(12046)
# 如果使用CPU，需要非常长的时间，建议减少模型规模来加速速度

In [20]:
# 一些超参数
emb_size = 128
head_size = 8
n_layer = 12
sequence_len = 64
learning_rate = 1e-3
eval_iters = 20
batch_size = 500
# 如果有GPU，该脚本
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [21]:
# 定义读取本地 .jsonl.gz 文件并创建 Dataset 的函数
def load_local_dataset(directory):
    all_data = []
    # 遍历目录下所有 .jsonl.gz 文件
    for filename in os.listdir(directory):
        if filename.endswith(".jsonl.gz"):
            file_path = os.path.join(directory, filename)
            with gzip.open(file_path, "rt", encoding="utf-8") as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        all_data.append(data)
                    except json.JSONDecodeError:
                        continue
    # 将列表转换为 Hugging Face Dataset
    return Dataset.from_list(all_data[:200])


# 本地数据所在的 train 目录路径
train_dir = "python_data/final/jsonl/train"
# 加载本地数据集
raw_datasets = load_local_dataset(train_dir)
datasets = raw_datasets

In [22]:
class CharTokenizer:

    def __init__(self, data, end_ind=0):
        # data:list[str]
        # 得到所有的字符
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in enumerate(chars)}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in self.char2ind.items()}
        self.end_ind = end_ind

    def encode(self, x):
        #x:str
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        #x:int or list[x]
        if isinstance(x, int):
            return self.ind2char[x]
        return [self.ind2char[i] for i in x]


tokenizer = CharTokenizer(datasets['original_string'])
test_str = 'def f(x)'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

[70, 71, 72, 2, 72, 10, 90, 11]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~─│└├'

In [23]:
def process(data, tokenizer, sequence_len=sequence_len):
    text = data['original_string']
    # text is list[str]
    inputs, labels = [], []
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        # 有bug，无法处理长度过小的数据
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i:i + sequence_len])
            labels.append(enc[i + 1:i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}


# 将数据分为训练集和测试集
tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)

f = lambda x: process(x, tokenizer)
# 创建了一个匿名函数，并将其赋值给变量f。这个函数接受一个参数x，并返回process(x, tokenizer)的执行结果。

tokenized = tokenized.map(f, batched=True, remove_columns=datasets.column_names)
tokenized.set_format(type='torch', device=device)

tokenized.shape['train'], tokenized.shape['test']

Map: 100%|██████████| 20/20 [00:00<00:00, 114.79 examples/s]


((159261, 2), (11307, 2))

In [24]:
train_loader = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=batch_size, shuffle=True)
next(iter(train_loader))

{'inputs': tensor([[73, 81, 84,  ..., 67, 78,  2],
         [ 2, 68, 71,  ..., 71, 70,  2],
         [79, 82, 78,  ..., 85, 16, 79],
         ...,
         [ 2,  2,  2,  ...,  2,  2,  2],
         [84, 52, 38,  ..., 84, 38, 71],
         [ 4, 11,  1,  ..., 71, 16, 73]], device='cuda:0'),
 'labels': tensor([[81, 84, 75,  ..., 78,  2, 72],
         [68, 71, 74,  ..., 70,  2, 86],
         [82, 78, 71,  ..., 16, 79, 67],
         ...,
         [ 2,  2,  2,  ...,  2,  2,  2],
         [52, 38, 38,  ..., 38, 71, 85],
         [11,  1,  2,  ..., 16, 73, 16]], device='cuda:0')}

In [25]:
@torch.no_grad()
def generate(model, context, tokenizer, max_new_tokens=300):
    # content:(1,T)
    # out = []
    out = context.tolist()[0]
    model.eval()
    for _ in range(max_new_tokens):
        # 由于注意力机制的长度限制，截断背景
        logits = model(context[:, -sequence_len:])
        probs = F.softmax(logits[:, -1, :], dim=-1)  # (1,98)probs 中的每个元素就代表了对应类别的预测概率。
        # 随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  # (1,1)这段代码会根据probs给定的概率分布进行随机抽样，返回被选中元素的索引
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [26]:
def estimate_loss(model):
    re = {}
    # 将模型切换至评估模式
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['test'] = _loss(model, test_loader)
    # 将模型切换至训练模式
    model.train()
    return re


@torch.no_grad()
def _loss(model, data_loader):
    """
    计算模型在不同数据集下面的评估指标
    """
    loss = []
    data_iter = iter(data_loader)
    # 随机使用多个批量数据来预估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None)
        if data is None:
            data_iter = iter(data_loader)
            data = next(data_iter, None)
        inputs, labels = data['inputs'], data['labels']  # (B,T)
        logits = model(inputs)
        loss.append(F.cross_entropy(logits.transpose(-2, -1), labels).item())
    return torch.tensor(loss).mean().item()

In [41]:
def train_model(model, optimizer, criterion=None, epochs=1):
    # 记录模型在训练集上的模型损失
    lossi = []
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels']  # (B,T)
            optimizer.zero_grad()
            logits = model(inputs)  # (B,T,vs)
            loss = F.cross_entropy(logits.transpose(-2, -1), labels)
            lossi.append(loss.item())
            print(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats["train"]:.4f}'
        test_loss = f'test loss {stats["test"]:.4f}'
        print(f'epoch {epoch:>2}: {train_loss:.4f} | {test_loss:.4f}')
    return lossi

In [28]:
def attention(query, key, value, dropout, mask=None):
    '''
    注意力机制
    参数
    ----
    query ：torch.FloatTensor，查询向量，形状为(B, T, C)
    key ：torch.FloatTensor，键向量，形状为(B, T, C)
    value ：torch.FloatTensor，数值向量，形状为(B, T, C)
    dropout ：随机失活
    mask ：torch.FloatTensor，掩码，形状为(T, T)
    返回
    ----
    out ：torch.FloatTensor，根据注意力机制得到的背景向量，形状为(B, T, C)
    w_att ：torch.FloatTensor，权重向量，形状为(B, T, T)
    '''
    # query, key, value都有相同的形状
    B, T, C = query.shape
    # (B, T, C) @ (B, C, T) --> (B, T, T)
    scores = query @ key.transpose(-2, -1) / (C ** 0.5)
    if mask is not None:
        # 如果没有mask，则表示词元可以使用左右两边的背景，也就是双向注意力
        # 如果mask是上三角矩阵，则表示自回归模式的单向注意力
        # mask的形状是(T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w_att = dropout(F.softmax(scores, dim=-1))  # (B, T, T)
    out = w_att @ value  # (B, T, C)
    return out, w_att

In [29]:
class MaskedAttention(nn.Module):

    def __init__(self, emb_size, head_size):
        '''
        单头单向注意力
        参数
        ----
        emb_size ：int，特征长度
        head_size ：int，背景向量长度
        '''
        super().__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        # 这个上三角矩阵不参与模型训练
        self.register_buffer(
            'tril', torch.tril(torch.ones(sequence_len, sequence_len)))
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.FloatTensor
            文本的特征向量，形状为(B, T, C)，其中B表示批量大小，T表示文本长度，C表示特征长度（emb_size）
        返回
        ----
        out ：torch.FloatTensor
            根据注意力机制得到的背景向量，形状为(B, T, H)，其中H表示背景向量长度（head_size）
        '''
        B, T, C = x.shape
        q = self.query(x)  # (B, T, H)
        k = self.key(x)    # (B, T, H)
        v = self.value(x)  # (B, T, H)
        mask = self.tril[:T, :T]
        out, _ = attention(q, k, v, self.dropout, mask)
        return out         # (B, T, H)

In [30]:
class MaskedMultiHeadAttention(nn.Module):

    def __init__(self, emb_size, head_size):
        '''
        多头单向注意力
        参数
        ----
        emb_size ：int，特征长度
        head_size ：int，背景向量长度
        '''
        super().__init__()
        # 确保特征长度是背景向量长度的倍数
        assert(emb_size % head_size == 0)
        # 定义单头注意力的个数
        n_head = emb_size // head_size
        heads = [MaskedAttention(emb_size, head_size) for _ in range(n_head)]
        self.heads = nn.ModuleList(heads)
        # 线性变换
        self.proj = nn.Linear(emb_size, emb_size)
        # 随机失活
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.FloatTensor
            文本的特征向量，形状为(B, T, C)，其中B表示批量大小，T表示文本长度，C表示特征长度（emb_size）
        返回
        ----
        out ：torch.FloatTensor，根据注意力机制得到的背景向量，形状为(B, T, C)
        '''
        # 将多个单头注意力的结果做张量拼接
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, C)
        out = self.dropout(self.proj(out))
        return out

In [31]:
class FeedForward(nn.Module):

    def __init__(self, emb_size):
        '''
        多层感知器
        '''
        super().__init__()
        self.l1 = nn.Linear(emb_size, 4 * emb_size)
        self.l2 = nn.Linear(4 * emb_size, emb_size)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        x = F.gelu(self.l1(x))
        out = self.dropout(self.l2(x))
        return out

In [32]:
class Block(nn.Module):

    def __init__(self, emb_size, head_size):
        '''
        解码块
        参数
        ----
        emb_size ：int，特征长度
        head_size ：int，单头注意力中的背景向量长度
        '''
        super().__init__()
        self.mha = MaskedMultiHeadAttention(emb_size, head_size)
        self.ff = FeedForward(emb_size)
        # 层归一化
        self.ln1 = nn.LayerNorm(emb_size)
        self.ln2 = nn.LayerNorm(emb_size)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.FloatTensor，文本的特征向量，形状为(B, T, C)
        返回
        ----
        out ：torch.FloatTensor，解码块的输出，形状为(B, T, C)
        '''
        # 残差连接
        x = x + self.mha(self.ln1(x))   # (B, T, C)
        out = x + self.ff(self.ln2(x))  # (B, T, C)
        return out

In [33]:
class CharGPT(nn.Module):

    def __init__(self, vs):
        '''
        利用GPT-2进行自然语言的自回归学习
        参数
        ----
        vs ：int，字典大小
        '''
        super().__init__()
        # 文字嵌入层
        self.token_embedding = nn.Embedding(vs, emb_size)
        # 位置嵌入层
        self.position_embedding = nn.Embedding(sequence_len, emb_size)
        # 解码块
        blocks = [Block(emb_size, head_size) for _ in range(n_layer)]
        self.blocks = nn.Sequential(*blocks)
        self.ln = nn.LayerNorm(emb_size)
        # 语言建模头
        self.lm_head = nn.Linear(emb_size, vs)

    def forward(self, x):
        '''
        向前传播
        参数
        ----
        x ：torch.LongTensor，当前字母在字典中的位置，形状为(B, T)
        返回
        ----
        logits ：torch.FloatTensor，预测结果的logits，形状为(B, T, vs)
        '''
        B, T = x.shape
        # 定义词元的位置，形状为(T)
        pos = torch.arange(0, T, dtype=torch.long, device=x.device)
        # 词元语义特征
        tok_emb = self.token_embedding(x)       # (B, T,  C)
        # 位置特征
        pos_emb = self.position_embedding(pos)  # (   T,  C)
        x = tok_emb + pos_emb                   # (B, T,  C)
        x = self.blocks(x)                      # (B, T,  C)
        x = self.ln(x)                          # (B, T,  C)
        logits = self.lm_head(x)                # (B, T, vs)
        return logits

In [34]:
c_model = CharGPT(len(tokenizer.char2ind)).to(device)
c_model

CharGPT(
  (token_embedding): Embedding(101, 128)
  (position_embedding): Embedding(64, 128)
  (blocks): Sequential(
    (0): Block(
      (mha): MaskedMultiHeadAttention(
        (heads): ModuleList(
          (0-15): 16 x MaskedAttention(
            (key): Linear(in_features=128, out_features=8, bias=False)
            (query): Linear(in_features=128, out_features=8, bias=False)
            (value): Linear(in_features=128, out_features=8, bias=False)
            (dropout): Dropout(p=0.4, inplace=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
      )
      (ff): FeedForward(
        (l1): Linear(in_features=128, out_features=512, bias=True)
        (l2): Linear(in_features=512, out_features=128, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affi

In [35]:
# 统计chatgpt有多少个参数
c_model, sum(p.numel() for p in c_model.parameters())

(CharGPT(
   (token_embedding): Embedding(101, 128)
   (position_embedding): Embedding(64, 128)
   (blocks): Sequential(
     (0): Block(
       (mha): MaskedMultiHeadAttention(
         (heads): ModuleList(
           (0-15): 16 x MaskedAttention(
             (key): Linear(in_features=128, out_features=8, bias=False)
             (query): Linear(in_features=128, out_features=8, bias=False)
             (value): Linear(in_features=128, out_features=8, bias=False)
             (dropout): Dropout(p=0.4, inplace=False)
           )
         )
         (proj): Linear(in_features=128, out_features=128, bias=True)
         (dropout): Dropout(p=0.4, inplace=False)
       )
       (ff): FeedForward(
         (l1): Linear(in_features=128, out_features=512, bias=True)
         (l2): Linear(in_features=512, out_features=128, bias=True)
         (dropout): Dropout(p=0.4, inplace=False)
       )
       (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
       (ln2): LayerNorm((128,), eps

In [36]:
context = torch.tensor(tokenizer.encode('def'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

def*ZOQh/of("YP{syE ?|dw│R<1'LS=├9NN[{/|=CK|AM:iKcaO;+Q3j<sA*gWS$M8Ax!8yT3zy'm5Za)'W*5\WmsB"B{<|e|>


In [37]:
estimate_loss(c_model)

{'train': 4.795285224914551, 'test': 4.806792259216309}

In [ ]:
l = train_model(c_model, optim.AdamW(c_model.parameters(), lr=learning_rate))

2.328467607498169
2.8299498558044434
2.4306836128234863
2.468092679977417
2.51009202003479
2.446960210800171
2.4664671421051025
2.441786050796509
2.42341685295105


In [ ]:
plt.plot(torch.tensor(l).view(-1, 10).mean(1).numpy())

In [ ]:
# 使用模型来生成文本
begin_text = torch.tensor(tokenizer.encode('def '), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, begin_text))))